In [1]:
#| default_exp core.transformers
#| export

In [2]:
#| export

import numpy as np

from tinytorch.core.activations import GELU
from tinytorch.core.attention import MultiHeadAttention
from tinytorch.core.autograd import Function
from tinytorch.core.embeddings import EmbeddingLayer
from tinytorch.core.layers import Linear

# Import from previous modules - following proper dependency chain
from tinytorch.core.tensor import Tensor

# Constants for memory calculations
BYTES_PER_FLOAT32 = 4  # Standard float32 size in bytes
MB_TO_BYTES = 1024 * 1024  # Megabytes to bytes conversion


def create_causal_mask(seq_len: int) -> Tensor:
    """
    Create a causal (autoregressive) attention mask.

    This mask ensures that position i can only attend to positions j where j ≤ i.
    Essential for autoregressive language models like GPT.

    Args:
        seq_len: Length of the sequence

    Returns:
        Tensor of shape (1, seq_len, seq_len) with:
        - 1.0 for positions that CAN be attended to (lower triangle)
        - 0.0 for positions that CANNOT be attended to (upper triangle)

    Example:
        For seq_len=4, creates:
        [[1, 0, 0, 0],
         [1, 1, 0, 0],
         [1, 1, 1, 0],
         [1, 1, 1, 1]]

    Usage:
        >>> from tinytorch.core.transformers import create_causal_mask
        >>> mask = create_causal_mask(seq_len=10)
        >>> output = attention(x, mask=mask)
    """
    # Lower triangular matrix: 1 = can attend, 0 = cannot attend
    mask = np.tril(np.ones((seq_len, seq_len), dtype=np.float32))
    return Tensor(mask[np.newaxis, :, :])  # Add batch dimension

In [4]:
#| export


class _LayerNormBackward(Function):
    """
    Gradient computation for the full layer normalization operation.

    Computes gradients for x, gamma, and beta in one pass.
    output = gamma * ((x - mean) / std) + beta

    The gradient for x uses the standard LayerNorm formula:
        dx = (gamma/std) * (grad - mean(grad) - normalized * mean(grad * normalized))
    """

    def __init__(self, x, gamma, beta, normalized_data, std_data):
        """Initialize with forward pass values needed for gradient computation."""
        super().__init__(x, gamma, beta)
        self.normalized_data = normalized_data
        self.std_data = std_data

    def apply(self, grad_output):
        """Compute gradients for LayerNorm (x, gamma, beta)."""
        x, gamma, beta = self.saved_tensors

        grad_x = grad_gamma = grad_beta = None
        normalized = self.normalized_data
        std_data = self.std_data

        # Gradient for beta: sum over all dims except last
        if isinstance(beta, Tensor) and beta.requires_grad:
            # Sum over batch and sequence dimensions
            grad_beta = grad_output.copy()
            while grad_beta.ndim > 1:
                grad_beta = grad_beta.sum(axis=0)

        # Gradient for gamma: sum of (grad_output * normalized) over batch/seq dims
        if isinstance(gamma, Tensor) and gamma.requires_grad:
            grad_gamma = (grad_output * normalized).copy()
            while grad_gamma.ndim > 1:
                grad_gamma = grad_gamma.sum(axis=0)

        # Gradient for x: full LayerNorm backward formula
        if isinstance(x, Tensor) and x.requires_grad:
            # grad flowing through gamma: grad_output * gamma
            gamma_data = gamma.data if isinstance(gamma, Tensor) else gamma
            grad_norm = grad_output * gamma_data

            mean_grad = np.mean(grad_norm, axis=-1, keepdims=True)
            mean_grad_norm = np.mean(grad_norm * normalized, axis=-1, keepdims=True)
            grad_x = (1.0 / std_data) * (grad_norm - mean_grad - normalized * mean_grad_norm)

        return (grad_x, grad_gamma, grad_beta)


class LayerNorm:
    """
    Layer Normalization for transformer blocks.

    Normalizes across the feature dimension (last axis) for each sample independently,
    unlike batch normalization which normalizes across the batch dimension.
    """

    def __init__(self, normalized_shape, eps=1e-5):
        """
        Initialize LayerNorm with learnable parameters.

        TODO: Set up normalization parameters

        APPROACH:
        1. Store the shape to normalize over (usually embed_dim)
        2. Initialize learnable scale (gamma) and shift (beta) parameters
        3. Set small epsilon for numerical stability

        EXAMPLE:
        >>> ln = LayerNorm(512)  # For 512-dimensional embeddings
        >>> x = Tensor(np.random.randn(2, 10, 512))  # (batch, seq, features)
        >>> normalized = ln.forward(x)
        >>> # Each (2, 10) sample normalized independently across 512 features

        HINTS:
        - gamma should start at 1.0 (identity scaling)
        - beta should start at 0.0 (no shift)
        - eps prevents division by zero in variance calculation
        """
        ### BEGIN SOLUTION
        self.normalized_shape = normalized_shape
        self.eps = eps

        # learnable parameters: scale and shift
        self.gamma = Tensor(np.ones(normalized_shape)) # scale parameter
        self.beta = Tensor(np.zeros(normalized_shape)) # shift parameter)
        ### END SOLUTION

    def forward(self, x):
        """
        Apply layer normalization.

        TODO: Implement layer normalization formula

        APPROACH:
        1. Compute mean and variance across the last dimension
        2. Normalize: (x - mean) / sqrt(variance + eps)
        3. Apply learnable scale and shift: gamma * normalized + beta

        MATHEMATICAL FORMULA:
        y = (x - μ) / σ * γ + β
        where μ = mean(x), σ = sqrt(var(x) + ε)

        HINT: Use keepdims=True to maintain tensor dimensions for broadcasting
        """
        ### BEGIN SOLUTION
        # Compute statistics across last dimension (features)
        mean_data = np.mean(x.data, axis=-1, keepdims=True)

        # compute variance: E[(x - μ)²]
        diff = x.data - mean_data
        variance = np.mean(diff * diff, axis=-1, keepdims=True)

        # normalize: (x - mean) / sqrt(variance + eps)
        std_data = np.sqrt(variance + self.eps)
        normalized_data = diff / std_data

        # apply learnable transformation: gamma * normalized + beta
        output_data = self.gamma.data * normalized_data + self.beta.data
        output = Tensor(output_data)

        # attach gradient function for full LayerNorm backward
        if x.requires_grad or self.gamma.requires_grad or self.beta.requires_grad:
            output.requires_grad = True
            output._grad_fn = _LayerNormBackward(
                x, self.gamma, self.beta, normalized_data, std_data
            )

        return output

        ### END SOLUTION

    def __call__(self, x):
        """Allows the layer norm to be called like a function."""
        return self.forward(x)

    def parameters(self):
        """Return learnable parameters."""
        return [self.gamma, self.beta]

In [5]:
def test_unit_layer_norm():
    """🧪 Test LayerNorm implementation."""
    print("🧪 Unit Test: Layer Normalization...")

    # Test basic normalization
    ln = LayerNorm(4)
    x = Tensor([[1.0, 2.0, 3.0, 4.0], [5.0, 6.0, 7.0, 8.0]])  # (2, 4)

    normalized = ln.forward(x)

    # Check output shape
    assert normalized.shape == (2, 4)

    # Check normalization properties (approximately)
    # For each sample, mean should be close to 0, std close to 1
    for i in range(2):
        sample_mean = np.mean(normalized.data[i])
        sample_std = np.std(normalized.data[i])
        assert abs(sample_mean) < 1e-5, f"Mean should be ~0, got {sample_mean}"
        assert abs(sample_std - 1.0) < 1e-4, f"Std should be ~1, got {sample_std}"

    # Test parameter shapes
    params = ln.parameters()
    assert len(params) == 2
    assert params[0].shape == (4,)  # gamma
    assert params[1].shape == (4,)  # beta

    print("✅ LayerNorm works correctly!")

# Run test immediately when developing this module
if __name__ == "__main__":
    test_unit_layer_norm()  # Moved after implementation

🧪 Unit Test: Layer Normalization...
✅ LayerNorm works correctly!
